# Test 4 — Cross-validation + Optuna

Combines the two previous ideas. For each model–target combination Optuna searches the
same space as Test 2 (`TPESampler(seed=42)`, `MedianPruner()`, 50 trials, objective =
minimise mean CV RMSE, `KFold(5, shuffle=True, random_state=42)`). The chosen parameters
are then evaluated by the **same 5-fold CV** used in Test 3, so Test 3 and Test 4 are
directly comparable — the only difference is tuned versus default.

Reads the shared `preprocessed.csv`.

In [1]:
# ============================================================
#  Imports and settings  (self-contained: this notebook needs
#  nothing from the other notebooks except preprocessed.csv)
# ============================================================
import json, math, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor,
                              AdaBoostRegressor, BaggingRegressor,
                              VotingRegressor, StackingRegressor)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import lightgbm as lgb

SEED = 42
TARGETS = ["DISINTEGRATION_TIME", "FRIABILITY", "Water absorption ratio"]

# ---- load the SAME preprocessed file every test notebook uses --------------
df = pd.read_csv("/content/preprocessed.csv")
print("preprocessed.csv :", df.shape)

def get_X_y(frame, target):
    """Features are every column except the three targets. Rows where the target
    itself is missing cannot be scored, so they are dropped for that target only."""
    drop = [c for c in TARGETS if c in frame.columns]
    m = frame[target].notna()
    X = frame.loc[m].drop(columns=drop).astype(float)
    y = frame.loc[m, target].astype(float)
    return X, y


# ============================================================
#  RobustScaler was chosen over StandardScaler on domain grounds:
#  the data carries genuine but skewed values and a few data-entry
#  outliers, and RobustScaler uses the median and IQR, so those do
#  not distort the scaling. Only the models that need scaling get it
#  (linear family, SVR, KNN); tree ensembles are scale-invariant.
# ============================================================
def scaled(est):
    # median-impute the few cells the anomaly rules repaired to NaN, then RobustScale
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", RobustScaler()), ("model", est)])

def imputed(est):
    # tree models that do not accept NaN get a median imputer only (no scaling needed)
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", est)])

def build_model(name, params=None):
    """The full regression zoo. `params` overrides defaults (used by the Optuna
    notebooks); when params is None the model runs at its default settings."""
    p = dict(params or {})
    if name == "LinearRegression":  return scaled(LinearRegression())
    if name == "Ridge":             return scaled(Ridge(random_state=SEED, **p))
    if name == "Lasso":             return scaled(Lasso(random_state=SEED, max_iter=5000, **p))
    if name == "ElasticNet":        return scaled(ElasticNet(random_state=SEED, max_iter=5000, **p))
    if name == "DecisionTree":      return imputed(DecisionTreeRegressor(random_state=SEED, **p))
    if name == "RandomForest":      return imputed(RandomForestRegressor(random_state=SEED, n_jobs=-1, **p))
    if name == "ExtraTrees":        return imputed(ExtraTreesRegressor(random_state=SEED, n_jobs=-1, **p))
    if name == "GradientBoosting":  return imputed(GradientBoostingRegressor(random_state=SEED, **p))
    if name == "HistGradientBoosting": return HistGradientBoostingRegressor(random_state=SEED, **p)
    if name == "AdaBoost":          return imputed(AdaBoostRegressor(random_state=SEED, **p))
    if name == "Bagging":           return imputed(BaggingRegressor(random_state=SEED, n_jobs=-1, **p))
    if name == "XGBoost":           return xgb.XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0, **p)
    if name == "LightGBM":          return lgb.LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1, **p)
    if name == "SVR":               return scaled(SVR(**p))
    if name == "KNN":               return scaled(KNeighborsRegressor(**p))
    if name == "VotingRegressor":
        # The paper's voting design: XGBoost + RandomForest + KNN + GradientBoosting, averaged.
        return VotingRegressor([
            ("xgb", xgb.XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0)),
            ("rf",  imputed(RandomForestRegressor(random_state=SEED, n_jobs=-1))),
            ("knn", scaled(KNeighborsRegressor(n_neighbors=2, weights="distance", metric="manhattan"))),
            ("gb",  imputed(GradientBoostingRegressor(random_state=SEED)))])
    if name == "StackingRegressor":
        return StackingRegressor(
            estimators=[
                ("xgb", xgb.XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0)),
                ("rf",  imputed(RandomForestRegressor(random_state=SEED, n_jobs=-1))),
                ("knn", scaled(KNeighborsRegressor(n_neighbors=2, weights="distance", metric="manhattan")))],
            final_estimator=imputed(GradientBoostingRegressor(random_state=SEED)), cv=5)
    raise ValueError(f"unknown model {name}")

# Every regression method we can reasonably fit ("all possible regression models").
ALL_MODELS = ["LinearRegression", "Ridge", "Lasso", "ElasticNet",
              "DecisionTree", "RandomForest", "ExtraTrees", "GradientBoosting",
              "HistGradientBoosting", "AdaBoost", "Bagging", "XGBoost", "LightGBM",
              "SVR", "KNN", "VotingRegressor", "StackingRegressor"]


# ============================================================
#  The five metrics asked for: MSE, MAE, RMSE, R2, Adjusted R2.
#  Adjusted R2 penalises R2 for the number of predictors p on n test rows:
#      1 - (1 - R2) * (n - 1) / (n - p - 1)
# ============================================================
def all_metrics(y_true, y_pred, n_features):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    n = len(y_true); p = n_features
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    denom = (n - p - 1)
    adj = 1 - (1 - r2) * (n - 1) / denom if denom > 0 else np.nan
    return dict(MSE=mse, MAE=mean_absolute_error(y_true, y_pred),
                RMSE=math.sqrt(mse), R2=r2, Adjusted_R2=adj)

preprocessed.csv : (1982, 79)


In [2]:
# ============================================================
#  Inference-vs-actual plot, printed inline (no image file saved).
#  This is the "graph that tells the performance of inferencing
#  results with actual data" from the plan: predicted on the y-axis,
#  actual on the x-axis, with the ideal y=x line.
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def show_inference_plot(results_dict, title):
    """results_dict maps target -> (y_true, y_pred)."""
    fig, axes = plt.subplots(1, len(results_dict), figsize=(5 * len(results_dict), 4))
    if len(results_dict) == 1:
        axes = [axes]
    for ax, (target, (yt, yp)) in zip(axes, results_dict.items()):
        ax.scatter(yt, yp, s=12, alpha=.5, edgecolor="none")
        lo, hi = min(np.min(yt), np.min(yp)), max(np.max(yt), np.max(yp))
        ax.plot([lo, hi], [lo, hi], "r--", lw=1, label="ideal (y = x)")
        ax.set_xlabel("actual"); ax.set_ylabel("predicted")
        ax.set_title(f"{target}\nbest model inference", fontsize=9)
        ax.legend(fontsize=7)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [3]:
!pip install optuna

In [4]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEARCH_SPACES = {
    "Ridge":        lambda t: dict(alpha=t.suggest_float("alpha", 1e-3, 1e3, log=True)),
    "Lasso":        lambda t: dict(alpha=t.suggest_float("alpha", 1e-4, 1e2, log=True)),
    "ElasticNet":   lambda t: dict(alpha=t.suggest_float("alpha", 1e-4, 1e2, log=True),
                                   l1_ratio=t.suggest_float("l1_ratio", 0.05, 0.99)),
    "DecisionTree": lambda t: dict(max_depth=t.suggest_int("max_depth", 2, 25),
                                   min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 40),
                                   min_samples_split=t.suggest_int("min_samples_split", 2, 40)),
    "RandomForest": lambda t: dict(n_estimators=t.suggest_int("n_estimators", 100, 500, step=50),
                                   max_depth=t.suggest_int("max_depth", 3, 25),
                                   min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 20),
                                   max_features=t.suggest_float("max_features", 0.1, 1.0)),
    "ExtraTrees":   lambda t: dict(n_estimators=t.suggest_int("n_estimators", 100, 500, step=50),
                                   max_depth=t.suggest_int("max_depth", 3, 25),
                                   min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 20),
                                   max_features=t.suggest_float("max_features", 0.1, 1.0)),
    "GradientBoosting": lambda t: dict(n_estimators=t.suggest_int("n_estimators", 100, 500, step=50),
                                   learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                                   max_depth=t.suggest_int("max_depth", 2, 8),
                                   subsample=t.suggest_float("subsample", 0.6, 1.0)),
    "HistGradientBoosting": lambda t: dict(learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                                   max_leaf_nodes=t.suggest_int("max_leaf_nodes", 7, 63),
                                   min_samples_leaf=t.suggest_int("min_samples_leaf", 5, 40),
                                   l2_regularization=t.suggest_float("l2_regularization", 1e-3, 10, log=True)),
    "AdaBoost":     lambda t: dict(n_estimators=t.suggest_int("n_estimators", 50, 300, step=50),
                                   learning_rate=t.suggest_float("learning_rate", 0.01, 2.0, log=True)),
    "Bagging":      lambda t: dict(n_estimators=t.suggest_int("n_estimators", 10, 100, step=10)),
    "XGBoost":      lambda t: dict(n_estimators=t.suggest_int("n_estimators", 100, 600, step=50),
                                   learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                                   max_depth=t.suggest_int("max_depth", 2, 10),
                                   subsample=t.suggest_float("subsample", 0.6, 1.0),
                                   colsample_bytree=t.suggest_float("colsample_bytree", 0.3, 1.0),
                                   reg_lambda=t.suggest_float("reg_lambda", 1e-3, 20, log=True)),
    "LightGBM":     lambda t: dict(n_estimators=t.suggest_int("n_estimators", 100, 600, step=50),
                                   learning_rate=t.suggest_float("learning_rate", 0.01, 0.3, log=True),
                                   num_leaves=t.suggest_int("num_leaves", 7, 63),
                                   min_child_samples=t.suggest_int("min_child_samples", 5, 40),
                                   subsample=t.suggest_float("subsample", 0.6, 1.0)),
    "SVR":          lambda t: dict(C=t.suggest_float("C", 0.1, 200, log=True),
                                   epsilon=t.suggest_float("epsilon", 0.01, 5, log=True),
                                   gamma=t.suggest_categorical("gamma", ["scale", "auto"])),
    "KNN":          lambda t: dict(n_neighbors=t.suggest_int("n_neighbors", 1, 20),
                                   weights=t.suggest_categorical("weights", ["uniform", "distance"]),
                                   metric=t.suggest_categorical("metric", ["euclidean", "manhattan", "minkowski"])),
}

CV = KFold(n_splits=5, shuffle=True, random_state=SEED)

def rmse_cv(model, X, y):
    neg = cross_val_score(model, X, y, cv=CV, scoring="neg_mean_squared_error", n_jobs=-1)
    return float(np.sqrt(-neg).mean())

def tune_one(name, X, y, n_trials=50):
    space = SEARCH_SPACES[name]
    def objective(trial):
        return rmse_cv(build_model(name, space(trial)), X, y)
    study = optuna.create_study(direction="minimize",
                                sampler=TPESampler(seed=SEED), pruner=MedianPruner())
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study

In [5]:
all_rows, best_infer = [], {}

for target in TARGETS:
    X, y = get_X_y(df, target)
    print(f"\n================  {target}  (n={len(y)})  ================")
    target_rows = []
    for name in ALL_MODELS:
        t0 = time.time()
        if name in SEARCH_SPACES:
            study = tune_one(name, X, y, n_trials=50)
            best_params = study.best_params
        else:
            best_params = {}
        # evaluate the tuned model by the SAME 5-fold CV used in Test 3
        fold_metrics = []
        for tr_idx, te_idx in CV.split(X):
            model = build_model(name, best_params).fit(X.iloc[tr_idx], y.iloc[tr_idx])
            pred = model.predict(X.iloc[te_idx])
            fold_metrics.append(all_metrics(y.iloc[te_idx], pred, X.shape[1]))
        fm = pd.DataFrame(fold_metrics)
        m = dict(target=target, model=name,
                 MSE=fm.MSE.mean(), MAE=fm.MAE.mean(), RMSE=fm.RMSE.mean(),
                 R2=fm.R2.mean(), R2_std=fm.R2.std(), Adjusted_R2=fm.Adjusted_R2.mean(),
                 params=json.dumps(best_params), seconds=round(time.time() - t0, 1))
        target_rows.append(m); all_rows.append(m)
        print(f"  {name:22s} R2={m['R2']:7.3f} (+/-{m['R2_std']:.3f})  "
              f"RMSE={m['RMSE']:9.3f}  ({m['seconds']}s)")
    best = max(target_rows, key=lambda r: r["R2"])
    oof = cross_val_predict(build_model(best["model"], json.loads(best["params"])),
                            X, y, cv=CV, n_jobs=-1)
    best_infer[target] = (y.to_numpy(), oof)
    print(f"  --> best tuned: {best['model']} (mean CV R2={best['R2']:.3f})")

results_rs4 = pd.DataFrame(all_rows)


================  DISINTEGRATION_TIME  (n=1982)  ================
  LinearRegression       R2=  0.273 (+/-0.089)  RMSE=   38.257  (1.1s)
  Ridge                  R2=  0.274 (+/-0.089)  RMSE=   38.216  (28.0s)
  Lasso                  R2=  0.274 (+/-0.089)  RMSE=   38.230  (110.4s)
  ElasticNet             R2=  0.274 (+/-0.089)  RMSE=   38.217  (114.4s)
  DecisionTree           R2=  0.491 (+/-0.122)  RMSE=   31.673  (13.0s)
  RandomForest           R2=  0.711 (+/-0.056)  RMSE=   23.954  (1270.7s)
  ExtraTrees             R2=  0.763 (+/-0.057)  RMSE=   21.587  (614.7s)
  GradientBoosting       R2=  0.754 (+/-0.052)  RMSE=   22.062  (1105.0s)
  HistGradientBoosting   R2=  0.742 (+/-0.068)  RMSE=   22.477  (160.9s)
  AdaBoost               R2=  0.369 (+/-0.118)  RMSE=   35.512  (387.6s)
  Bagging                R2=  0.704 (+/-0.053)  RMSE=   24.301  (571.2s)
  XGBoost                R2=  0.757 (+/-0.049)  RMSE=   21.934  (843.3s)
  LightGBM               R2=  0.751 (+/-0.068)  RMSE=   22.

### Results table (tuned, mean over 5 folds)

In [6]:
print(results_rs4[["target","model","MSE","MAE","RMSE","R2","R2_std","Adjusted_R2"]]
      .round(3).to_string(index=False))

                target                model      MSE    MAE   RMSE     R2  R2_std  Adjusted_R2
   DISINTEGRATION_TIME     LinearRegression 1469.928 23.798 38.257  0.273   0.089        0.100
   DISINTEGRATION_TIME                Ridge 1466.572 23.734 38.216  0.274   0.089        0.102
   DISINTEGRATION_TIME                Lasso 1467.588 23.752 38.230  0.274   0.089        0.101
   DISINTEGRATION_TIME           ElasticNet 1466.594 23.736 38.217  0.274   0.089        0.102
   DISINTEGRATION_TIME         DecisionTree 1005.230 16.009 31.673  0.491   0.122        0.370
   DISINTEGRATION_TIME         RandomForest  574.081 11.870 23.954  0.711   0.056        0.642
   DISINTEGRATION_TIME           ExtraTrees  467.633 10.089 21.587  0.763   0.057        0.707
   DISINTEGRATION_TIME     GradientBoosting  487.368 10.892 22.062  0.754   0.052        0.696
   DISINTEGRATION_TIME HistGradientBoosting  506.025 11.688 22.477  0.742   0.068        0.681
   DISINTEGRATION_TIME             AdaBoost 1266.2

### Best tuned model per target

In [7]:
best_rs4 = results_rs4.loc[results_rs4.groupby("target")["R2"].idxmax()]
print(best_rs4[["target","model","RMSE","R2","R2_std","Adjusted_R2"]].round(3).to_string(index=False))
for _, r in best_rs4.iterrows():
    print(f"\n{r['target']} -> {r['model']}\n  best params: {r['params']}")

                target      model   RMSE    R2  R2_std  Adjusted_R2
   DISINTEGRATION_TIME ExtraTrees 21.587 0.763   0.057        0.707
            FRIABILITY        SVR  0.899 0.133   0.153       -0.074
Water absorption ratio ExtraTrees  9.554 0.947   0.018        0.934

DISINTEGRATION_TIME -> ExtraTrees
  best params: {"n_estimators": 250, "max_depth": 25, "min_samples_leaf": 1, "max_features": 0.7981051758729255}

FRIABILITY -> SVR
  best params: {"C": 0.9779891777372498, "epsilon": 0.10315591374863017, "gamma": "auto"}

Water absorption ratio -> ExtraTrees
  best params: {"n_estimators": 250, "max_depth": 24, "min_samples_leaf": 1, "max_features": 0.7840093790192766}


### Inference vs actual (out-of-fold predictions of best tuned model)

In [8]:
show_inference_plot(best_infer, "Test 4 (CV + Optuna) — out-of-fold predicted vs actual")

### Save this result set, then compare all four

The block below loads the four result CSVs (run Tests 1–4 first) and prints the best model
per target in each, side by side, plus the overall best. It also lines them up against the
research paper's reported numbers.

In [9]:
results_rs4.to_csv("results_test4_cv_optuna.csv", index=False)
print("saved results_test4_cv_optuna.csv")

# ---- final comparison across all four result sets --------------------------
import os
frames = {}
for tag, fn in [("Test1_baseline", "results_test1_baseline.csv"),
                ("Test2_optuna", "results_test2_optuna.csv"),
                ("Test3_cv", "results_test3_cv.csv"),
                ("Test4_cv_optuna", "results_test4_cv_optuna.csv")]:
    if os.path.exists(fn):
        frames[tag] = pd.read_csv(fn)

if len(frames) == 4:
    print("\n================  BEST MODEL PER TARGET, EACH RESULT SET  ================")
    summary = []
    for tag, fr in frames.items():
        b = fr.loc[fr.groupby("target")["R2"].idxmax()]
        for _, r in b.iterrows():
            summary.append(dict(result_set=tag, target=r["target"],
                                 model=r["model"], R2=r["R2"], RMSE=r["RMSE"]))
    summ = pd.DataFrame(summary)
    print(summ.pivot_table(index="target", columns="result_set", values="R2").round(3).to_string())

    print("\n================  OVERALL BEST (across all four result sets)  ================")
    overall = summ.loc[summ.groupby("target")["R2"].idxmax()]
    print(overall[["target","result_set","model","R2","RMSE"]].round(3).to_string(index=False))
    overall.to_csv("BEST_overall_model.csv", index=False)

    print("\n================  COMPARISON WITH THE RESEARCH PAPER  ================")
    paper = pd.DataFrame([
        dict(target="DISINTEGRATION_TIME",    paper_model="Voting (tuned)", paper_R2=0.76, paper_RMSE=21.99),
        dict(target="FRIABILITY",             paper_model="RandomForest",   paper_R2=0.70, paper_RMSE=0.142),
        dict(target="Water absorption ratio", paper_model="KNN k=2",        paper_R2=0.94, paper_RMSE=10.07)])
    cmp = paper.merge(overall[["target","result_set","model","R2","RMSE"]], on="target")
    cmp["R2_gain_vs_paper"] = (cmp.R2 - cmp.paper_R2).round(3)
    print(cmp.round(3).to_string(index=False))
    cmp.to_csv("COMPARISON_vs_paper.csv", index=False)
else:
    print(f"\n(only {len(frames)} of 4 result files found — run Tests 1-3 first, "
          f"then this cell will print the full comparison.)")

saved results_test4_cv_optuna.csv

(only 1 of 4 result files found — run Tests 1-3 first, then this cell will print the full comparison.)


In [10]:
# ============================================================
#  ACTUAL vs PREDICTED — FULL preprocessed dataset (all rows)
#  Reusable across Test 1, 2, 3, 4. Change the two settings.
# ============================================================
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_predict

# ---- SETTINGS: change these two lines per notebook --------------------------
RESULTS_CSV       = "/content/results_test4_cv_optuna.csv"   # -> _test2_optuna / _test3_cv / _test4_cv_optuna
PREPROCESSED_CSV  = "/content/preprocessed.csv"               # the full cleaned dataset, all 1982 rows
MODE               = "cv"   # "holdout" for Test 1 & Test 2  |  "cv" for Test 3 & Test 4

results = pd.read_csv(RESULTS_CSV)
full_df = pd.read_csv(PREPROCESSED_CSV)   # read fresh — independent of whatever `df` is in memory


def get_best_row(results, target):
    sub = results[results.target == target]
    return sub.loc[sub.R2.idxmax()]


def summarise(actual, predicted, n_features):
    """MSE, MAE, RMSE, R2, Adjusted R2 on whatever subset is passed in."""
    n = len(actual)
    r2 = 1 - ((actual - predicted) ** 2).sum() / ((actual - actual.mean()) ** 2).sum()
    mse = ((actual - predicted) ** 2).mean()
    mae = (actual - predicted).abs().mean()
    denom = n - n_features - 1
    adj_r2 = 1 - (1 - r2) * (n - 1) / denom if denom > 0 else np.nan
    return dict(n=n, MSE=mse, MAE=mae, RMSE=np.sqrt(mse), R2=r2, Adjusted_R2=adj_r2)


def metrics_on_full_dataset(target, mode=MODE):
    row = get_best_row(results, target)
    model_name = row["model"]
    params = json.loads(row["params"]) if ("params" in row and pd.notna(row.get("params"))) else {}

    X, y = get_X_y(full_df, target)
    n_feat = X.shape[1]

    if mode == "holdout":
        # Same split the test notebook used, so the model is the same one that was reported.
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=SEED)
        model = build_model(model_name, params).fit(X_tr, y_tr)
        pred = model.predict(X)                       # predict EVERY row in preprocessed.csv
        split_label = np.where(X.index.isin(X_tr.index), "train", "test")
    elif mode == "cv":
        # cross_val_predict already gives an out-of-fold prediction for every row —
        # this IS the honest full-dataset version, no train/test contamination.
        cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
        pred = cross_val_predict(build_model(model_name, params), X, y, cv=cv, n_jobs=-1)
        split_label = np.array(["out-of-fold"] * len(X))
    else:
        raise ValueError("mode must be 'holdout' or 'cv'")

    out = pd.DataFrame({"row_index": X.index, "target": target, "model": model_name,
                        "split": split_label, "actual": y.to_numpy(), "predicted": pred})
    out["abs_error"] = (out.actual - out.predicted).abs()
    out["pct_error"] = 100 * out.abs_error / out.actual.replace(0, np.nan)

    full_m = summarise(out.actual, out.predicted, n_feat)
    if mode == "holdout":
        train_m = summarise(out.loc[out.split == "train", "actual"], out.loc[out.split == "train", "predicted"], n_feat)
        test_m  = summarise(out.loc[out.split == "test",  "actual"], out.loc[out.split == "test",  "predicted"], n_feat)
    else:
        train_m = test_m = None   # no train/test split to break out under CV

    return out, full_m, train_m, test_m


all_tables, metric_rows = [], []
for t in TARGETS:
    tbl, full_m, train_m, test_m = metrics_on_full_dataset(t)
    print(f"\n{t}  ({tbl.model.iloc[0]})  —  first 10 rows")
    print(tbl.head(10).round(3).to_string(index=False))

    rows_to_show = [("FULL preprocessed.csv", full_m)]
    if train_m is not None:
        rows_to_show += [("train rows only", train_m), ("test rows only", test_m)]
    for label, m in rows_to_show:
        print(f"  {label:24s} n={m['n']:4d}  MSE={m['MSE']:.3f}  MAE={m['MAE']:.3f}  "
              f"RMSE={m['RMSE']:.3f}  R2={m['R2']:.3f}  AdjR2={m['Adjusted_R2']:.3f}")
        metric_rows.append(dict(target=t, model=tbl.model.iloc[0], subset=label, **m))
    all_tables.append(tbl)

actual_vs_pred_full = pd.concat(all_tables, ignore_index=True)
metrics_summary = pd.DataFrame(metric_rows)

out1 = RESULTS_CSV.replace("results_", "actual_vs_predicted_FULLDATA_")
out2 = RESULTS_CSV.replace("results_", "metrics_FULLDATA_")
actual_vs_pred_full.to_csv(out1, index=False)
metrics_summary.to_csv(out2, index=False)
print(f"\nsaved {out1}  ({len(actual_vs_pred_full)} rows)")
print(f"saved {out2}")

# Display results in the terminal/output cell without saving CSV files
# print("\n" + "=" * 100)
# print("ACTUAL VS PREDICTED — FULL DATASET")
# print("=" * 100)
# print(actual_vs_pred_full.round(3).to_string(index=False))

# print("\n" + "=" * 100)
# print("METRICS SUMMARY")
# print("=" * 100)
# print(metrics_summary.round(3).to_string(index=False))


DISINTEGRATION_TIME  (ExtraTrees)  —  first 10 rows
 row_index              target      model       split  actual  predicted  abs_error  pct_error
         0 DISINTEGRATION_TIME ExtraTrees out-of-fold     9.5     32.706     23.206    244.276
         1 DISINTEGRATION_TIME ExtraTrees out-of-fold   372.0    168.035    203.965     54.829
         2 DISINTEGRATION_TIME ExtraTrees out-of-fold   190.0    171.617     18.383      9.675
         3 DISINTEGRATION_TIME ExtraTrees out-of-fold   102.0    243.086    141.086    138.319
         4 DISINTEGRATION_TIME ExtraTrees out-of-fold   252.0    287.423     35.423     14.057
         5 DISINTEGRATION_TIME ExtraTrees out-of-fold   218.0    254.145     36.145     16.580
         6 DISINTEGRATION_TIME ExtraTrees out-of-fold   307.0    244.000     63.000     20.521
         7 DISINTEGRATION_TIME ExtraTrees out-of-fold   280.0    269.208     10.792      3.854
         8 DISINTEGRATION_TIME ExtraTrees out-of-fold    16.0     16.424      0.424      2.6

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# SETTINGS
# ============================================================
RESULTS_CSV = "/content/actual_vs_predicted_FULLDATA_test4_cv_optuna.csv"
MODE = "cv"   # "holdout" for Test 1/2, "cv" for Test 3/4

# Automatically find the prediction CSV created by your code
PREDICTIONS_CSV = RESULTS_CSV.replace(
    "results_",
    "actual_vs_predicted_FULLDATA_"
)

# Read predictions
actual_vs_pred_full = pd.read_csv(PREDICTIONS_CSV)

print("File successfully loaded:", PREDICTIONS_CSV)
print("Shape:", actual_vs_pred_full.shape)
display(actual_vs_pred_full.head())


# ============================================================
# PREPARE DATA FOR PLOTTING
# ============================================================
if MODE == "holdout":
    # Only unseen test rows
    plot_df = actual_vs_pred_full[
        actual_vs_pred_full["split"] == "test"
    ].copy()
else:
    # All predictions are out-of-fold under CV
    plot_df = actual_vs_pred_full.copy()

plot_df["residual"] = (
    plot_df["actual"] - plot_df["predicted"]
)

print("\nRows being plotted:", len(plot_df))
print("Targets:", plot_df["target"].unique())


# ============================================================
# CREATE GRAPHS
# ============================================================
sns.set_theme(style="whitegrid")

for target in plot_df["target"].unique():

    d = plot_df[plot_df["target"] == target].copy()

    if d.empty:
        print(f"No rows available for target: {target}")
        continue

    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    fig.suptitle(
        f"Target: {target}\nModel: {d['model'].iloc[0]}",
        fontsize=16,
        fontweight="bold"
    )

    # 1. Actual vs predicted scatter plot
    sns.scatterplot(
        data=d,
        x="actual",
        y="predicted",
        alpha=0.7,
        ax=axes[0, 0]
    )

    minimum = min(d["actual"].min(), d["predicted"].min())
    maximum = max(d["actual"].max(), d["predicted"].max())

    axes[0, 0].plot(
        [minimum, maximum],
        [minimum, maximum],
        "r--",
        linewidth=2,
        label="Perfect prediction"
    )

    axes[0, 0].set_title("Actual vs Predicted")
    axes[0, 0].legend()

    # 2. Residual plot
    sns.scatterplot(
        data=d,
        x="predicted",
        y="residual",
        alpha=0.7,
        ax=axes[0, 1]
    )

    axes[0, 1].axhline(0, color="red", linestyle="--")
    axes[0, 1].set_title("Residual Plot")
    axes[0, 1].set_ylabel("Actual − Predicted")

    # 3. Residual distribution
    sns.histplot(
        data=d,
        x="residual",
        bins=30,
        kde=True,
        ax=axes[1, 0]
    )

    axes[1, 0].axvline(0, color="red", linestyle="--")
    axes[1, 0].set_title("Residual Distribution")

    # 4. Sorted actual vs predicted
    d_sorted = d.sort_values("actual").reset_index(drop=True)

    axes[1, 1].plot(
        d_sorted.index,
        d_sorted["actual"],
        label="Actual",
        color="blue",
        linewidth=2
    )

    axes[1, 1].plot(
        d_sorted.index,
        d_sorted["predicted"],
        label="Predicted",
        color="orange",
        linewidth=2
    )

    axes[1, 1].set_title("Sorted Actual vs Predicted")
    axes[1, 1].set_xlabel("Observations")
    axes[1, 1].legend()

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

File successfully loaded: /content/actual_vs_predicted_FULLDATA_test4_cv_optuna.csv
Shape: (5946, 8)


,row_index,target,model,split,actual,predicted,abs_error,pct_error
0,0,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,9.5,32.706199,23.206199,244.275776
1,1,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,372.0,168.034777,203.965223,54.829361
2,2,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,190.0,171.617109,18.382891,9.675206
3,3,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,102.0,243.085600,141.085600,138.319216
4,4,DISINTEGRATION_TIME,ExtraTrees,out-of-fold,252.0,287.423360,35.423360,14.056889



Rows being plotted: 5946
Targets: ['DISINTEGRATION_TIME' 'FRIABILITY' 'Water absorption ratio']
